# 02 — All-source acquisition and corpus integrity gate

Notebook 02 streams every enabled dataset source, admits every usable data class, verifies that each required task family has train/validation/test coverage, and validates independent splits. Bootstrap and synthetic records are included when their payloads are usable. Metadata-only catalogues remain separate until their media payload can be decoded.

In [ ]:
import os, sys, json
from pathlib import Path

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "configs" / "phone-research.yaml").is_file() and (candidate / "src" / "onecore").is_dir():
            return candidate
    raise RuntimeError(f"Could not locate OneCore project root from {current}")

root = find_project_root()
print("OneCore project root:", root)
sys.path.insert(0, str(root / "src"))
from onecore.acquire import acquire_sources, assert_task_coverage
from onecore.data import validate_training_data, validate_validation_data, validate_test_data, validate_split_disjointness

report_path = root / "data" / "acquisition-report.json"
if os.environ.get("ONECORE_SKIP_ACQUISITION", "0") == "1":
    if not report_path.is_file():
        raise FileNotFoundError("restored corpus is missing acquisition-report.json")
    acquisition_report = json.loads(report_path.read_text(encoding="utf-8"))
else:
    acquisition_report = acquire_sources(root, profile=os.environ.get("ONECORE_DATA_PROFILE", "initial"))

required = {item.strip() for item in os.environ.get("ONECORE_REQUIRED_TASKS", "text").split(",") if item.strip()}
if required:
    coverage = assert_task_coverage(
        root, required,
        minimum_train=int(os.environ.get("ONECORE_MIN_TRAIN_RECORDS", "25")),
        minimum_holdout=int(os.environ.get("ONECORE_MIN_HOLDOUT_RECORDS", "5")),
    )
    acquisition_report["task_coverage"] = coverage
    report_path.write_text(json.dumps(acquisition_report, ensure_ascii=False, indent=2), encoding="utf-8")

print("Acquisition summary:", json.dumps({
    "source_count": acquisition_report.get("source_count"),
    "new_records": acquisition_report.get("new_records"),
    "total_new_records": acquisition_report.get("total_new_records"),
    "task_counts": acquisition_report.get("task_counts"),
    "task_coverage": acquisition_report.get("task_coverage"),
    "completed_sources": acquisition_report.get("completed_sources"),
    "unavailable_sources": acquisition_report.get("unavailable_sources"),
    "metadata_only_sources": acquisition_report.get("metadata_only_sources"),
    "sources_with_access_required": acquisition_report.get("sources_with_access_required"),
    "sources": [{"source": item.get("source"), "status": item.get("status"), "read": item.get("read"), "accepted": item.get("accepted"), "errors": item.get("errors")} for item in acquisition_report.get("sources", [])],
}, ensure_ascii=False, indent=2))

train_report = validate_training_data(root)
validation_report = validate_validation_data(root)
test_report = validate_test_data(root)
disjointness_report = validate_split_disjointness(root)
report = {"train": train_report, "validation": validation_report, "test": test_report, "split_disjointness": disjointness_report}
print(json.dumps(report, ensure_ascii=False, indent=2))
print("Dataset gates: PASS — all usable data classes are included; task coverage and split separation verified")
